# 🚀 Z-Image-Turbo (Versão Final Corrigida)
Carregamento direto na GPU para evitar erro de memória (OOM).

### 🔑 Configuração dos Secrets:
Certifique-se de que `HF_TOKEN` e `NGROK_TOKEN` estão configurados no painel lateral.

In [ ]:
# 1. Instalação das dependências
!pip install -U git+https://github.com/huggingface/diffusers git+https://github.com/Disty0/sdnq
!pip install fastapi uvicorn pillow pyngrok huggingface_hub python-multipart accelerate

In [ ]:
# 2. Autenticação
import os, torch, diffusers
from google.colab import userdata
from huggingface_hub import login
from pyngrok import ngrok
from sdnq.loader import apply_sdnq_options_to_model

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: login(token=hf_token)
    ngrok_token = userdata.get('NGROK_TOKEN')
    if ngrok_token: ngrok.set_auth_token(ngrok_token)
    print('✅ Autenticação realizada!')
except Exception as e:
    print(f'⚠️ Erro: {e}')

In [ ]:
# 3. Backend FastAPI com Carregamento Seguro
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
import base64, time, uvicorn, threading
from io import BytesIO

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

print('⏳ Carregando pesos diretamente na GPU...')
model_id = 'Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32'

pipe = diffusers.ZImagePipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map='cuda'
)

pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)

print('✅ Modelo pronto!')

class GenerateRequest(BaseModel):
    prompt: str
    steps: int = 9
    aspect_ratio: str = '1:1'

ASPECT_RATIOS = {'1:1': (1024, 1024), '16:9': (1216, 704), '9:16': (704, 1216)}

@app.post('/generate')
async def generate(req: GenerateRequest):
    start_time = time.time()
    w, h = ASPECT_RATIOS.get(req.aspect_ratio, (1024, 1024))
    try:
        with torch.no_grad():
            output = pipe(prompt=req.prompt, height=h, width=w, num_inference_steps=req.steps, guidance_scale=0.0).images[0]
        buffered = BytesIO()
        output.save(buffered, format='JPEG', quality=85)
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return {'image': f'data:image/jpeg;base64,{img_str}', 'latency': f'{time.time() - start_time:.2f}s'}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def start(): uvicorn.run(app, host='0.0.0.0', port=8000)
threading.Thread(target=start, daemon=True).start()
time.sleep(10)
public_url = ngrok.connect(8000).public_url
print(f'\n🚀 BACKEND ONLINE: {public_url}')